In [42]:
import random
import sympy as sp
import h5py
import torch
import torch.nn as nn
import numpy as np
from typing import List, Dict, Tuple, Union, Optional

## Class to handle the tokenizing of an amplitude

In [70]:
class ScatteringAmplitudeTokenizer:
    """
    Tokenizer for scattering amplitude expressions involving momenta p_i, 
    polarization vectors e_i, and field strengths F.
    """
    def __init__(self, max_particles: int = 8, max_sequence_length: int = 2048): # 8 particles by default
        self.max_particles = max_particles
        self.max_sequence_length = max_sequence_length
        
        # Define initial hard-coded vocabulary (added "0:" for digit-level encoding)
        self.vocab_init = {
            "<PAD>": 0,
            "<UNK>": 1,
            "<BOS>": 2,
            "<EOS>": 3,
            "+": 4,
            "-": 5,
            "*": 6,
            "/": 7,
            "^": 8,
            "(": 9,
            ")": 10,
            "0:": 11,
            "1:": 12,
            "2:": 13,
            "3:": 14,
            "4:": 15,
            "5:": 16,
            "6:": 17,
            "7:": 18,
            "8:": 19,
            "9:": 20,
            "10:": 21,
            "·": 22,  # dot operation    
        }
        
        self.index = len(self.vocab_init)
        
        # Momentum tokens: p_1, p_2, ..., p_n
        self.p_tokens = {f"p_{i}": i + self.index for i in range(1, max_particles + 1)}
        
        # Polarization tokens: e_1, e_2, ..., e_n
        self.e_tokens = {f"e_{i}": i + self.index + max_particles for i in range(1, max_particles + 1)}
        
        # Field strength tokens: F_1, F_2, ..., F_n
        self.f_tokens = {f"F_{i}": i + self.index + 2 * max_particles for i in range(1, max_particles + 1)}
        
        # Build the full vocabulary
        self.vocab = {**self.vocab_init, **self.p_tokens, **self.e_tokens, **self.f_tokens}
        
        # Inverse mapping
        self.id_to_token = {v: k for k, v in self.vocab.items()}
        
        self.vocab_size = len(self.vocab)
    
    def tokenize(self, expression: str) -> List[int]:
        """
        Convert a string representation of a scattering amplitude expression to tokens.
        
        Example: "p_1·p_2 + e_1·p_2 + p_1·F_3·p_2"
        """
        # Preprocessing: Normalize the expression
        expression = expression.replace(" ", "")
        
        # Split the expression at operators while keeping them
        parts = []
        current = ""
        for char in expression:
            if char in "+-*()^":
                if current:
                    parts.append(current)
                parts.append(char)
                current = ""
            else:
                current += char
        if current:
            parts.append(current)
        
        # Process each term
        tokens = [self.vocab["<BOS>"]]
        
        for part in parts:
            if part in "+-*()^":
                tokens.append(self.vocab[part])
            else:
                # Check if the part is a number
                if part.isdigit():
                    # If number is between 1 and 10, use the pre-defined token
                    if 1 <= int(part) <= 10:
                        tokens.append(self.vocab[f"{part}:"])
                    else:
                        # For numbers outside our vocabulary, split into digit tokens.
                        for digit in part:
                            key = f"{digit}:"
                            if key in self.vocab:
                                tokens.append(self.vocab[key])
                            else:
                                tokens.append(self.vocab["<UNK>"])
                else:
                    # Handle dot product terms like "p_1·p_2" or "p_1·F_3·p_2"
                    subparts = part.split("·")
                    for i, subpart in enumerate(subparts):
                        if subpart in self.vocab:
                            tokens.append(self.vocab[subpart])
                        else:
                            # Try to match with known patterns
                            found = False
                            for prefix in ["p_", "e_", "F_"]:
                                if subpart.startswith(prefix):
                                    idx = subpart[len(prefix):]
                                    token_key = f"{prefix}{idx}"
                                    if token_key in self.vocab:
                                        tokens.append(self.vocab[token_key])
                                        found = True
                                        break
                            if not found:
                                tokens.append(self.vocab["<UNK>"])
                        
                        # Add dot token between subparts
                        if i < len(subparts) - 1:
                            tokens.append(self.vocab["·"])
        
        tokens.append(self.vocab["<EOS>"])
        
        # Ensure the sequence doesn't exceed max length
        if len(tokens) > self.max_sequence_length:
            tokens = tokens[:self.max_sequence_length - 1] + [self.vocab["<EOS>"]]
        
        return tokens
    
    def pad_sequence(self, tokens: List[int]) -> List[int]:
        """Pad or truncate a token sequence to max_sequence_length"""
        if len(tokens) < self.max_sequence_length:
            tokens = tokens + [self.vocab["<PAD>"]] * (self.max_sequence_length - len(tokens))
        else:
            tokens = tokens[:self.max_sequence_length]
        return tokens
    
    def batch_encode(self, expressions: List[str]) -> torch.Tensor:
        """Tokenize and pad a batch of expressions"""
        batch_tokens = [self.pad_sequence(self.tokenize(expr)) for expr in expressions]
        return torch.tensor(batch_tokens, dtype=torch.long)
    
    def decode(self, tokens: List[int]) -> str:
        """Convert tokens back to a string expression with reduced whitespace."""
        # Convert token ids to strings (strip trailing colons for digit tokens)
        decoded = []
        for token in tokens:
            if token in self.id_to_token and token not in [self.vocab["<PAD>"], 
                                                        self.vocab["<BOS>"], 
                                                        self.vocab["<EOS>"]]:
                token_str = self.id_to_token[token]
                if token_str.endswith(":") and token_str[:-1].isdigit():
                    decoded.append(token_str[:-1])
                else:
                    decoded.append(token_str)
        
        # Merge consecutive digit tokens into single numbers
        merged_tokens = []
        number_buffer = ""
        for tok in decoded:
            if tok.isdigit():
                number_buffer += tok
            else:
                if number_buffer:
                    merged_tokens.append(number_buffer)
                    number_buffer = ""
                merged_tokens.append(tok)
        if number_buffer:
            merged_tokens.append(number_buffer)
        
        # Join tokens with a single space and remove spaces around dot operator, multiplication and power
        expression = " ".join(merged_tokens)
        expression = expression.replace(" · ", "·").replace(" * ", "*").replace(" ^ ", "^").replace("( ", "(").replace(" )", ")")
        
        return expression.strip()



## Test

In [74]:
tokenizer = ScatteringAmplitudeTokenizer()
print(tokenizer.vocab)
print(tokenizer.vocab_size)

# Example expressions
expressions = [
    "p_1·p_2",
    "e_1·p_2",
    "e_1·e_2",
    "p_1·F_3·p_2",
    "p_1·F_3·F_4·p_2",
    "p_1·F_3·F_4·F_5·p_2",
    "(p_1·p_2 + 3*e_1·p_2 + 123*p_1·F_3·p_2)^124",
    "(8*p_1·p_2 + 3*e_1·p_2 + 57*p_1·F_3·F_2·F_3·F_4·p_2)^8",
]

# Tokenize examples
for expr in expressions:
    tokens = tokenizer.tokenize(expr)
    decoded = tokenizer.decode(tokens)
    print(f"Original: {expr}")
    print(f"Tokens: {tokens}")
    print(f"Decoded: {decoded}")
    print()

{'<PAD>': 0, '<UNK>': 1, '<BOS>': 2, '<EOS>': 3, '+': 4, '-': 5, '*': 6, '/': 7, '^': 8, '(': 9, ')': 10, '0:': 11, '1:': 12, '2:': 13, '3:': 14, '4:': 15, '5:': 16, '6:': 17, '7:': 18, '8:': 19, '9:': 20, '10:': 21, '·': 22, 'p_1': 24, 'p_2': 25, 'p_3': 26, 'p_4': 27, 'p_5': 28, 'p_6': 29, 'p_7': 30, 'p_8': 31, 'e_1': 32, 'e_2': 33, 'e_3': 34, 'e_4': 35, 'e_5': 36, 'e_6': 37, 'e_7': 38, 'e_8': 39, 'F_1': 40, 'F_2': 41, 'F_3': 42, 'F_4': 43, 'F_5': 44, 'F_6': 45, 'F_7': 46, 'F_8': 47}
47
Original: p_1·p_2
Tokens: [2, 24, 22, 25, 3]
Decoded: p_1·p_2

Original: e_1·p_2
Tokens: [2, 32, 22, 25, 3]
Decoded: e_1·p_2

Original: e_1·e_2
Tokens: [2, 32, 22, 33, 3]
Decoded: e_1·e_2

Original: p_1·F_3·p_2
Tokens: [2, 24, 22, 42, 22, 25, 3]
Decoded: p_1·F_3·p_2

Original: p_1·F_3·F_4·p_2
Tokens: [2, 24, 22, 42, 22, 43, 22, 25, 3]
Decoded: p_1·F_3·F_4·p_2

Original: p_1·F_3·F_4·F_5·p_2
Tokens: [2, 24, 22, 42, 22, 43, 22, 44, 22, 25, 3]
Decoded: p_1·F_3·F_4·F_5·p_2

Original: (p_1·p_2 + 3*e_1·p_2 + 